In [1]:
#Import libraries
import numpy as np
import pandas as pd
import seaborn as sb
import matplotlib.pyplot as plt 
sb.set() 
import re

In [2]:
#train set
train_data = pd.read_csv('train.csv')
train_data.head()

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0


In [3]:
#test set
test_data = pd.read_csv("test.csv")
test_data.head()

,id,comment_text
0,00001cee341fdb12,Yo bitch Ja Rule is more succesful then you'll...
1,0000247867823ef7,== From RfC == \n\n The title is fine as it is...
2,00013b17ad220c46,""" \n\n == Sources == \n\n * Zawe Ashton on Lap..."
3,00017563c3f7919a,":If you have a look back at the source, the in..."
4,00017695ad8997eb,I don't anonymously edit articles at all.


In [4]:
# train data stats
train_data.describe()

,toxic,severe_toxic,obscene,threat,insult,identity_hate
count,159571.000000,159571.000000,159571.000000,159571.000000,159571.000000,159571.000000
mean,0.095844,0.009996,0.052948,0.002996,0.049364,0.008805
std,0.294379,0.099477,0.223931,0.054650,0.216627,0.093420
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [5]:
# test data stats
test_data.describe()

,id,comment_text
count,153164,153164
unique,153164,153164
top,00001cee341fdb12,Yo bitch Ja Rule is more succesful then you'll...
freq,1,1


In [6]:
# train dataset info
train_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 159571 entries, 0 to 159570
Data columns (total 8 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   id             159571 non-null  object
 1   comment_text   159571 non-null  object
 2   toxic          159571 non-null  int64 
 3   severe_toxic   159571 non-null  int64 
 4   obscene        159571 non-null  int64 
 5   threat         159571 non-null  int64 
 6   insult         159571 non-null  int64 
 7   identity_hate  159571 non-null  int64 
dtypes: int64(6), object(2)
memory usage: 9.7+ MB


In [7]:
# test dataset info
test_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 153164 entries, 0 to 153163
Data columns (total 2 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   id            153164 non-null  object
 1   comment_text  153164 non-null  object
dtypes: object(2)
memory usage: 2.3+ MB


In [8]:
print("Train shape:", train_data.shape)
print("Test shape:", test_data.shape)

Train shape: (159571, 8)
Test shape: (153164, 2)


In [9]:
print("\nTrain columns:")
print(train_data.columns)

print("\nTest columns:")
print(test_data.columns)


Train columns:
Index(['id', 'comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat',
       'insult', 'identity_hate'],
      dtype='object')

Test columns:
Index(['id', 'comment_text'], dtype='object')


# DATA PREPROCESSING/CLEANING
- Filling missing values✔️ 
- Text Cleaning✔️ 
- TF-IDF with removal of stopwards ✔️ 
- Stemming(impacts TF-IDF performance)❌

In [10]:
# check for null values (missing comments)
print("Missing values in train set:")
print(train_data.isnull().sum())

print("\nMissing values in test set:")
print(test_data.isnull().sum())

Missing values in train set:
id               0
comment_text     0
toxic            0
severe_toxic     0
obscene          0
threat           0
insult           0
identity_hate    0
dtype: int64

Missing values in test set:
id              0
comment_text    0
dtype: int64


In [11]:
#fill null values/missing comments in case of any
train_data["comment_text"] = train_data["comment_text"].fillna("")
test_data["comment_text"] = test_data["comment_text"].fillna("")

In [12]:
#create a text cleaning function to clean raw text

def clean_text(text):
    text = str(text)
    text = text.lower()
    
    # remove HTML tags
    text = re.sub(r'<.*?>', ' ', text)
    
    # expand contractions
    text = re.sub(r"what's", "what is ", text)
    text = re.sub(r"\'s", " ", text)
    text = re.sub(r"\'ve", " have ", text)
    text = re.sub(r"can't", "can not ", text)
    text = re.sub(r"n't", " not ", text)
    text = re.sub(r"i'm", "i am ", text)
    text = re.sub(r"\'re", " are ", text)
    text = re.sub(r"\'d", " would ", text)
    text = re.sub(r"\'ll", " will ", text)

    # remove numbers
    text = re.sub(r'\b\w*\d+\w*\b', ' ', text)

    # remove punctuation
    text = re.sub(r'\W', ' ', text)

    # remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

In [13]:
#apply text cleaning
train_data["clean_comment"] = train_data["comment_text"].apply(clean_text)
test_data["clean_comment"] = test_data["comment_text"].apply(clean_text)

In [14]:
#cleaned textual train data
train_data[["comment_text", "clean_comment"]].head(10)

,comment_text,clean_comment
0,Explanation\nWhy the edits made under my usern...,explanation why the edits made under my userna...
1,D'aww! He matches this background colour I'm s...,d aww he matches this background colour i am s...
2,"Hey man, I'm really not trying to edit war. It...",hey man i am really not trying to edit war it ...
3,"""\nMore\nI can't make any real suggestions on ...",more i can not make any real suggestions on im...
4,"You, sir, are my hero. Any chance you remember...",you sir are my hero any chance you remember wh...
5,"""\n\nCongratulations from me as well, use the ...",congratulations from me as well use the tools ...
6,COCKSUCKER BEFORE YOU PISS AROUND ON MY WORK,cocksucker before you piss around on my work
7,Your vandalism to the Matt Shirvington article...,your vandalism to the matt shirvington article...
8,Sorry if the word 'nonsense' was offensive to ...,sorry if the word nonsense was offensive to yo...
9,alignment on this subject and which are contra...,alignment on this subject and which are contra...


In [15]:
#cleaned textual test data
test_data[["comment_text", "clean_comment"]].head(10)

,comment_text,clean_comment
0,Yo bitch Ja Rule is more succesful then you'll...,yo bitch ja rule is more succesful then you wi...
1,== From RfC == \n\n The title is fine as it is...,from rfc the title is fine as it is imo
2,""" \n\n == Sources == \n\n * Zawe Ashton on Lap...",sources zawe ashton on lapland
3,":If you have a look back at the source, the in...",if you have a look back at the source the info...
4,I don't anonymously edit articles at all.,i do not anonymously edit articles at all
5,Thank you for understanding. I think very high...,thank you for understanding i think very highl...
6,Please do not add nonsense to Wikipedia. Such ...,please do not add nonsense to wikipedia such e...
7,:Dear god this site is horrible.,dear god this site is horrible
8,""" \n Only a fool can believe in such numbers. ...",only a fool can believe in such numbers the co...
9,== Double Redirects == \n\n When fixing double...,double redirects when fixing double redirects ...


In [16]:
#Final check for any missing values
print("Empty comments after cleaning:",
      (train_data["clean_comment"] == "").sum())
print('\n')

#Checking text length distribution
train_data["clean_comment_length"] = train_data["clean_comment"].apply(len)
train_data["clean_comment_length"].describe()
print('\n')

#Vocabulary check of text data
sample_words = " ".join(train_data["clean_comment"].head(1000)).split()
print(sample_words[:50])

Empty comments after cleaning: 11




['explanation', 'why', 'the', 'edits', 'made', 'under', 'my', 'username', 'hardcore', 'metallica', 'fan', 'were', 'reverted', 'they', 'were', 'not', 'vandalisms', 'just', 'closure', 'on', 'some', 'gas', 'after', 'i', 'voted', 'at', 'new', 'york', 'dolls', 'fac', 'and', 'please', 'do', 'not', 'remove', 'the', 'template', 'from', 'the', 'talk', 'page', 'since', 'i', 'am', 'retired', 'now', 'd', 'aww', 'he', 'matches']


In [17]:
#Separation of text and labels in train set inorder to perform TF-IDF
label_cols = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]
X_train_text = train_data["clean_comment"]   #text from train set
y_train = train_data[label_cols]             #labels from train set


X_test_text = test_data["clean_comment"]      #text from test set

In [18]:
#Performing TF-IDF on text data of train & test set to convert textual data to numerical data

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    stop_words='english',
    min_df=2,
    max_df=0.9
)
# this to be used for machine learning*
X_train_tfidf = tfidf.fit_transform(X_train_text) #fit and transform data
X_test_tfidf = tfidf.transform(X_test_text)   #only transform test data based on vocab learnt during fitting train data 

In [19]:
#viewing TF-IDF results
print("X_train_tfidf shape:", X_train_tfidf.shape)  
print("X_test_tfidf shape:", X_test_tfidf.shape)

print("\nFirst 20 TF-IDF features:")
print(tfidf.get_feature_names_out()[:20])

X_train_tfidf shape: (159571, 20000)
X_test_tfidf shape: (153164, 20000)

First 20 TF-IDF features:
['__' 'aa' 'aap' 'aaron' 'ab' 'abandon' 'abandoned' 'abbey' 'abbreviation'
 'abbreviations' 'abc' 'abd' 'abdul' 'abide' 'abilities' 'ability'
 'ability create' 'ability edit' 'able' 'able edit']


The TF-IDF feature names are derived from the training data only, as the vectorizer is fitted on the training set. The test data is transformed using the same vocabulary without introducing new features

In [20]:
#saving cleaned train & test textual data in new csv files
train_data.to_csv("train_cleaned.csv", index=False)
test_data.to_csv("test_cleaned.csv", index=False)

#### Variables to be used for Logistic Regression
X_train_tfidf, y_train, X_test_tfidf, y_test         

X_train_tfidf and y_train will be used to train the Logistic Regression model. 

X_test_tfidf to be used to predict based on trained model and y_test for evaluating performance.